# Community detection on the creator crossover network

**Purpose**: does the creator-crossover structure across all 23 tracked titles resolve into empirically-detected communities that match genre, platform, or region — or something else entirely?

**Updated 2026-09-20, two follow-ups on the original run**: (1) publisher is now measured directly at the pair level (same-publisher vs. different-publisher `enrichment_ratio`), not just eyeballed as a modal value inside an already-detected community. (2) The region variable has been **replaced**, not just re-labeled: the original tournament-country-derived region turned out dominated by wherever grassroots tournament *volume* concentrates on Liquipedia (18 of 23 titles landed on "Asia-Pacific"), a real, flagged weakness in the first version. This version derives region from `language_mix_snapshots` instead — audience-side, not competitive-scene-side.

**Updated again same day, two corrections and one new test**: (1) the Read section previously claimed every internal pair in the fighting and mobile communities cleared `enrichment_ratio > 1.0`. That was false — checked and fixed directly against a new internal-pairs table (below every community listing), not re-asserted in prose. (2) Shared-channel counts are now reported alongside every enrichment ratio, everywhere communities are discussed — the mobile cluster's enriched pairs run on 2-7 shared channels, the fighting cluster's on 15-71, League/Teamfight Tactics on 1,278; that range is real information the ratio alone hides. (3) A new, direct mobile-vs-PC crossover test (not inferred from Community 2's platform purity) compares mobile-mobile, pc-pc, and mobile-pc pairs directly, with an explicit caveat that Twitch under-represents mobile audiences relative to PC.

**Refreshed again same day, against fully current data**: this repo's local database was 101 commits behind `origin/main` for most of this notebook's history; pulling and reloading brought it from 205,717 to 233,540 channels. Community membership is unchanged; every specific number in the Read section below (internal-pair fractions, publisher/platform means, the raw-vs-rank Adjusted Rand Index) has been recomputed against current data and corrected where it moved — see the Read section itself for what changed and by how much. `creator_crossover_null_model_test.ipynb` also now exists and is the authoritative test of whether any of these four communities are statistically real; its finding (Communities 1-3 real, Community 0 not) supersedes this notebook's own informal "real cluster vs. residual structure" language.

**Refreshed again 2026-10-02, against the current ~5-week window**: 296,583 channels, up from 233,540. Raw-weight community membership is unchanged. Two notable moves: Community 3 (fighting) is now a fully internally-coherent cluster (6 of 6 pairs positively enriched, up from 5 of 6 -- Guilty Gear/Mortal Kombat crossed the baseline), while Community 2 (mobile) lost two of its previously-clearing pairs (now 5 of 10, down from 6 of 10) as they settled back toward chance with more data. The raw-vs-rank Adjusted Rand Index rose again, from 0.407 to 0.535 -- the mobile cluster's rank-weighted partition is now an exact match to its raw-weighted one, stronger than the previous 'survives, diluted' result. `creator_crossover_null_model_test.ipynb` was also re-run; Communities 1-3 remain statistically real (Community 3 strengthened to z=14.97), Community 0 still is not (z=-0.27). See the Read section for the full breakdown.

**A note on the request as given**: there is no `creator_crossover_history` table in `etl/schema.sql` — `creator_crossover.ipynb` computes `enrichment_ratio` live, once, per run, and never persists it. "Most recent run" in practice means "computed fresh against the live database right now," which is what this notebook does — same query and same `enrichment_ratio`/hypergeometric method as `creator_crossover.ipynb` and `creator_crossover_subset.ipynb`, extended to all 23 titles rather than a 7-title subset, not pulled from a history table that doesn't exist.

**Method**: `networkx`'s built-in `louvain_communities` (networkx >= 3.0 ships Louvain natively) rather than the separate `python-louvain` package or `leidenalg`/`igraph` for Leiden — neither of the latter two was already installed, and Leiden's marginal benefit over Louvain isn't worth a heavier native-dependency chain (igraph) for a demonstrative run like this one. (Leiden was in fact added afterward, in a separate notebook, `creator_crossover_leiden.ipynb` — see that notebook for the direct comparison.) `networkx>=3.2` added to `requirements.txt` under the notebooks section (analysis-only, never imported by a collector script, so no risk to CLAUDE.md's "one rule" collectors).

**The caveat requested directly, stated up front rather than left to the read section**: 236 of 253 possible title pairs have `enrichment_ratio <= 1.0` — at or below what chance would predict (updated 2026-10-02, was 240; 17 pairs now show real positive enrichment, up from 13). **Community detection here is finding relative degrees of mutual avoidance, not positive affinity** — Louvain doesn't know the difference, it just finds the partition that maximizes modularity given whatever weights it's handed. Run twice to check this isn't an artifact of the raw ratio's scale: once on raw `enrichment_ratio` as edge weight, once on the same edges reweighted by rank (removing the outsized influence of the handful of pairs with enrichment >> 1). Compared with an Adjusted Rand Index (computed by hand below — not worth adding `scikit-learn` as a dependency for one function).

In [1]:
# Imports and repo path setup -- same pattern as every other notebook.
import sys
from itertools import combinations
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").is_file():
            return candidate
    raise RuntimeError("Could not find repo root (CLAUDE.md not found in any parent directory) -- run this notebook from somewhere inside the repo")

REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import yaml
import networkx as nx
from scipy.stats import rankdata, hypergeom

from etl.db import get_connection

In [2]:
# All 23 active titles, and their genre/platform/publisher -- config/titles.yaml
# is the source of truth, same as every other notebook in this project.
with open(REPO_ROOT / "config" / "titles.yaml") as f:
    titles_config = yaml.safe_load(f)["titles"]

active_titles = [t for t in titles_config if t.get("is_active")]
display_name_by_id = {t["id"]: t["display_name"] for t in active_titles}
genre_by_id = {t["id"]: t.get("genre") for t in active_titles}
platform_by_id = {t["id"]: t.get("platform") for t in active_titles}
publisher_by_id = {t["id"]: t.get("publisher") for t in active_titles}
all_title_ids = sorted(display_name_by_id.keys())

print(f"{len(all_title_ids)} active titles")

23 active titles


In [3]:
# Enrichment ratio, all 253 possible pairs -- same method as
# creator_crossover.ipynb / creator_crossover_subset.ipynb, computed fresh
# (no creator_crossover_history table exists to read a "most recent run"
# from -- see intro). Hypergeometric p-value added alongside (needed for
# the mobile-vs-PC test later in this notebook, and useful context
# everywhere else a shared/enrichment pair is reported).
def hypergeom_pvalue(shared: int, total_a: int, total_b: int, population: int) -> float:
    """One-sided p-value in whichever direction the overlap actually went --
    same method as creator_crossover_subset.ipynb / region_language_confound.ipynb."""
    if total_a == 0 or total_b == 0:
        return float("nan")
    dist = hypergeom(population, total_a, total_b)
    expected = total_a * total_b / population
    return float(dist.sf(shared - 1)) if shared >= expected else float(dist.cdf(shared))


conn = get_connection()
raw_rows = conn.execute(
    "SELECT channel_id, title_id, viewer_count FROM viewership_snapshots WHERE is_official_broadcast = 0"
).fetchall()
conn.close()

raw_df = pd.DataFrame(raw_rows, columns=["channel_id", "title_id", "viewer_count"])
channel_title_df = raw_df.groupby(["channel_id", "title_id"])["viewer_count"].sum().rename("viewer_time").reset_index()

titles_per_channel = channel_title_df.groupby("channel_id")["title_id"].nunique()
crossover_channel_ids = titles_per_channel[titles_per_channel > 1].index
crossover_df = channel_title_df[channel_title_df["channel_id"].isin(crossover_channel_ids)]

n_total_channels = channel_title_df["channel_id"].nunique()
total_channels_by_title = channel_title_df.groupby("title_id")["channel_id"].nunique()

titles_by_channel = crossover_df.groupby("channel_id")["title_id"].apply(set)
pair_channels: dict[tuple[str, str], set] = {}
for channel_id, title_set in titles_by_channel.items():
    for a, b in combinations(sorted(title_set), 2):
        pair_channels.setdefault((a, b), set()).add(channel_id)

pair_rows = []
for a, b in combinations(all_title_ids, 2):
    shared = len(pair_channels.get((a, b), set()))
    total_a, total_b = total_channels_by_title.get(a, 0), total_channels_by_title.get(b, 0)
    expected = total_a * total_b / n_total_channels if n_total_channels else float("nan")
    enrichment = shared / expected if expected > 0 else float("nan")
    p = hypergeom_pvalue(shared, total_a, total_b, n_total_channels)
    pair_rows.append({"a": a, "b": b, "shared": shared, "enrichment": enrichment, "p": p})

pair_df = pd.DataFrame(pair_rows)
n_with_edge = (pair_df["shared"] > 0).sum()
n_enriched = (pair_df["enrichment"] > 1.0).sum()
print(f"{n_total_channels} distinct above-threshold, non-official channels across all 23 titles")
print(f"{n_with_edge} of {len(pair_df)} possible pairs have >=1 shared crossover channel (a real edge)")
print(f"{n_enriched} of {len(pair_df)} pairs show enrichment_ratio > 1.0 (positive affinity, not suppression)")
pair_df.sort_values("enrichment", ascending=False).head(15)

296583 distinct above-threshold, non-official channels across all 23 titles
209 of 253 possible pairs have >=1 shared crossover channel (a real edge)
17 of 253 pairs show enrichment_ratio > 1.0 (positive affinity, not suppression)


,a,b,shared,enrichment,p
143,guilty_gear,street_fighter,56,5.482247,3.436906e-24
16,age_of_empires_ii,starcraft2,11,5.117687,1.465788e-05
145,guilty_gear,tekken,35,5.010276,1.949523e-14
195,mortal_kombat,tekken,26,4.391100,6.849911e-10
186,mobile_legends_bb,wild_rift,12,3.276593,4.032384e-04
244,street_fighter,tekken,132,2.960945,3.193315e-27
120,free_fire,mobile_legends_bb,6,2.959753,1.721929e-02
132,free_fire,wild_rift,3,2.739510,9.823100e-02
178,mobile_legends_bb,pubg_mobile,12,2.728120,1.908474e-03
171,league_of_legends,teamfight_tactics,1960,2.504849,0.000000e+00


In [4]:
# Build the weighted graph -- nodes = 23 titles, edges = enrichment_ratio.
# Pairs with zero shared channels get NO edge (not a zero-weight edge) --
# for modularity-based community detection the two are equivalent, but
# "no edge" is the honest representation of "never observed together."
G_raw = nx.Graph()
G_raw.add_nodes_from(all_title_ids)
edge_pairs = []  # keep the exact edge order so the rank-weighted graph below reweights the SAME edge set
for _, row in pair_df[pair_df["shared"] > 0].iterrows():
    G_raw.add_edge(row["a"], row["b"], weight=row["enrichment"])
    edge_pairs.append((row["a"], row["b"], row["enrichment"]))

print(f"Graph: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges")
print(f"Isolated nodes (no crossover with anything): {list(nx.isolates(G_raw))}")

Graph: 23 nodes, 209 edges
Isolated nodes (no crossover with anything): []


In [5]:
# Louvain on raw enrichment_ratio weights. Fixed seed for reproducibility
# (Louvain's local-move order is randomized; a different seed can find a
# different, similarly-good local optimum on a network with weak signal
# like this one -- worth knowing, not hidden).
LOUVAIN_SEED = 42
communities_raw = nx.algorithms.community.louvain_communities(G_raw, weight="weight", seed=LOUVAIN_SEED)
communities_raw = sorted(communities_raw, key=len, reverse=True)

print(f"{len(communities_raw)} communities found (raw enrichment weights):\n")
for i, comm in enumerate(communities_raw):
    names = sorted(display_name_by_id[t] for t in comm)
    print(f"Community {i} ({len(comm)} titles): {', '.join(names)}")

modularity_raw = nx.algorithms.community.modularity(G_raw, communities_raw, weight="weight")
print(f"\nModularity: {modularity_raw:.3f}")

4 communities found (raw enrichment weights):

Community 0 (8 titles): Apex Legends, Fortnite, League of Legends, Overwatch, Rainbow Six Siege, Rocket League, Teamfight Tactics, VALORANT
Community 1 (6 titles): Age of Empires II, Counter-Strike 2, Dota 2, Hearthstone, PUBG: BATTLEGROUNDS, StarCraft II
Community 2 (5 titles): Brawl Stars, Free Fire, League of Legends: Wild Rift, Mobile Legends: Bang Bang, PUBG Mobile
Community 3 (4 titles): Guilty Gear -Strive-, Mortal Kombat 1, Street Fighter 6, Tekken 8

Modularity: 0.439


In [6]:
# Internal pairs of every community, with shared-channel counts and
# enrichment_ratio side by side -- correction (2026-09-20): the first
# version of this notebook's Read section claimed every internal pair in
# the fighting and mobile communities cleared enrichment_ratio > 1.0. Not
# true (mortal_kombat/street_fighter = 0.867, guilty_gear/mortal_kombat =
# 0, several mobile-cluster pairs below 1.0 too) -- computed and printed
# directly below instead of asserted in prose, so this can't silently
# drift out of sync with the data again. Shared-channel counts shown
# alongside the ratio throughout: a 5x enrichment on 7 shared channels
# (Age of Empires II/StarCraft II) and a 4x enrichment on 21 (Guilty Gear/
# Tekken) are not the same strength of evidence, and the ratio alone
# hides that.
def internal_pairs_report(communities: list[set], pair_df: pd.DataFrame, label: str) -> None:
    for i, comm in enumerate(communities):
        if len(comm) < 2:
            continue
        sub = pair_df[pair_df["a"].isin(comm) & pair_df["b"].isin(comm)].copy()
        sub["title_a"] = sub["a"].map(display_name_by_id)
        sub["title_b"] = sub["b"].map(display_name_by_id)
        sub = sub.sort_values("enrichment", ascending=False)
        n_clear = (sub["enrichment"] > 1.0).sum()
        print(f"{label} Community {i} ({len(comm)} titles) -- {n_clear} of {len(sub)} internal pairs clear the chance baseline:")
        print(sub[["title_a", "title_b", "shared", "enrichment", "p"]].to_string(index=False))
        print()

internal_pairs_report(communities_raw, pair_df, "Raw-weight")

Raw-weight Community 0 (8 titles) -- 1 of 28 internal pairs clear the chance baseline:
          title_a           title_b  shared  enrichment             p
League of Legends Teamfight Tactics    1960    2.504849  0.000000e+00
Teamfight Tactics          VALORANT    1082    0.867486  3.709984e-08
Rainbow Six Siege     Rocket League     557    0.605984  5.807443e-42
League of Legends          VALORANT    4423    0.554683  0.000000e+00
     Apex Legends         Overwatch    1691    0.495064 6.676133e-283
     Apex Legends          VALORANT    2692    0.459641  0.000000e+00
         Fortnite     Rocket League    1618    0.457240  0.000000e+00
        Overwatch          VALORANT    3304    0.445710  0.000000e+00
     Apex Legends Teamfight Tactics     253    0.440212  1.354310e-56
        Overwatch Teamfight Tactics     292    0.401414  2.126463e-84
    Rocket League          VALORANT    1086    0.325329  0.000000e+00
League of Legends         Overwatch    1493    0.321042  0.000000e+00
   

In [7]:
# Same edge set, reweighted by RANK instead of raw enrichment_ratio -- does
# the outsized magnitude of a few pairs (Age of Empires II/StarCraft II at
# ~5x) drive the partition, or does the ordering alone reproduce it?
weights_only = [w for _, _, w in edge_pairs]
ranks = rankdata(weights_only)  # 1 = lowest enrichment, N = highest

G_rank = nx.Graph()
G_rank.add_nodes_from(all_title_ids)
for (a, b, _), r in zip(edge_pairs, ranks):
    G_rank.add_edge(a, b, weight=float(r))

communities_rank = nx.algorithms.community.louvain_communities(G_rank, weight="weight", seed=LOUVAIN_SEED)
communities_rank = sorted(communities_rank, key=len, reverse=True)

print(f"{len(communities_rank)} communities found (rank-based weights):\n")
for i, comm in enumerate(communities_rank):
    names = sorted(display_name_by_id[t] for t in comm)
    print(f"Community {i} ({len(comm)} titles): {', '.join(names)}")

modularity_rank = nx.algorithms.community.modularity(G_rank, communities_rank, weight="weight")
print(f"\nModularity: {modularity_rank:.3f}")

3 communities found (rank-based weights):

Community 0 (10 titles): Apex Legends, Fortnite, Guilty Gear -Strive-, Mortal Kombat 1, Overwatch, Rainbow Six Siege, Rocket League, Street Fighter 6, Tekken 8, VALORANT
Community 1 (8 titles): Age of Empires II, Counter-Strike 2, Dota 2, Hearthstone, League of Legends, PUBG: BATTLEGROUNDS, StarCraft II, Teamfight Tactics
Community 2 (5 titles): Brawl Stars, Free Fire, League of Legends: Wild Rift, Mobile Legends: Bang Bang, PUBG Mobile

Modularity: 0.145


In [8]:
# Same internal-pairs report for the rank-weighted partition -- "wherever
# communities are reported" per the request, not just the raw-weight run.
internal_pairs_report(communities_rank, pair_df, "Rank-weight")

Rank-weight Community 0 (10 titles) -- 6 of 45 internal pairs clear the chance baseline:
             title_a              title_b  shared  enrichment             p
Guilty Gear -Strive-     Street Fighter 6      56    5.482247  3.436906e-24
Guilty Gear -Strive-             Tekken 8      35    5.010276  1.949523e-14
     Mortal Kombat 1             Tekken 8      26    4.391100  6.849911e-10
    Street Fighter 6             Tekken 8     132    2.960945  3.193315e-27
     Mortal Kombat 1     Street Fighter 6      16    1.847979  1.530726e-02
Guilty Gear -Strive-      Mortal Kombat 1       2    1.474158  3.933974e-01
        Apex Legends     Street Fighter 6     321    0.765326  7.525611e-08
   Rainbow Six Siege        Rocket League     557    0.605984  5.807443e-42
        Apex Legends            Overwatch    1691    0.495064 6.676133e-283
     Mortal Kombat 1        Rocket League      15    0.472416  5.767399e-04
        Apex Legends             VALORANT    2692    0.459641  0.000000e+00

In [9]:
# Adjusted Rand Index between the two partitions, computed directly rather
# than adding scikit-learn as a dependency for one function. Standard
# formula: build the contingency table between the two label sets, then
# ARI = (sum_comb - expected) / (max_possible - expected).
from math import comb

def labels_from_communities(communities, node_order):
    label_by_node = {}
    for i, comm in enumerate(communities):
        for node in comm:
            label_by_node[node] = i
    return [label_by_node[n] for n in node_order]

def adjusted_rand_index(labels_a, labels_b):
    df = pd.DataFrame({"a": labels_a, "b": labels_b})
    contingency = pd.crosstab(df["a"], df["b"]).values
    n = contingency.sum()
    sum_comb_c = sum(comb(int(x), 2) for x in contingency.sum(axis=1))
    sum_comb_k = sum(comb(int(x), 2) for x in contingency.sum(axis=0))
    sum_comb = sum(comb(int(x), 2) for x in contingency.flatten())
    expected = sum_comb_c * sum_comb_k / comb(int(n), 2)
    max_index = 0.5 * (sum_comb_c + sum_comb_k)
    if max_index == expected:
        return 1.0  # degenerate case (e.g. everything in one cluster both times)
    return (sum_comb - expected) / (max_index - expected)

labels_raw = labels_from_communities(communities_raw, all_title_ids)
labels_rank = labels_from_communities(communities_rank, all_title_ids)
ari = adjusted_rand_index(labels_raw, labels_rank)
print(f"Adjusted Rand Index (raw-weight partition vs. rank-weight partition): {ari:.3f}")
print("(1.0 = identical partitions, 0.0 = no more agreement than chance, negative = worse than chance)")

Adjusted Rand Index (raw-weight partition vs. rank-weight partition): 0.535
(1.0 = identical partitions, 0.0 = no more agreement than chance, negative = worse than chance)


In [10]:
# Audience-region profile per title -- REPLACES the tournament-country-
# derived region used in the first version of this notebook. That version
# turned out dominated by wherever grassroots/regional-qualifier tournament
# *volume* concentrates on Liquipedia (18 of 23 titles landed on
# "Asia-Pacific"), not where a title's audience actually is -- flagged
# directly as a caveat there, fixed here rather than left as a caveat to
# repeat forever. This version is audience-side: `language_mix_snapshots`
# (viewer language mix, `verified` confidence, Twitch's own per-stream
# field -- see etl/schema.sql), the same table `niche_membership.ipynb` and
# `title_geography_maps.ipynb`'s Map 2 already use for audience-language
# work, extended here to all 23 titles rather than a handful.
#
# LANGUAGE_TO_REGION reuses title_geography_maps.ipynb's own vetted
# LANGUAGE_TO_COUNTRY (only languages with one genuinely dominant country
# included -- en/es/zh/ar/other stay excluded as too ambiguous to place on
# one country/region), bucketed up to the same macro-regions used for the
# (now-retired) tournament-region variable so the two are comparable at
# the same granularity.
LANGUAGE_TO_REGION = {
    "ko": "Asia-Pacific", "ja": "Asia-Pacific", "th": "Asia-Pacific", "vi": "Asia-Pacific",
    "id": "Asia-Pacific", "tl": "Asia-Pacific", "hi": "Asia-Pacific", "ms": "Asia-Pacific",
    "he": "MENA",
    "ru": "CIS", "uk": "CIS",
    "de": "Europe", "fr": "Europe", "it": "Europe", "pl": "Europe", "tr": "Europe",
    "cs": "Europe", "hu": "Europe", "el": "Europe", "fi": "Europe", "sv": "Europe",
    "da": "Europe", "no": "Europe", "nl": "Europe", "ro": "Europe", "bg": "Europe",
    "sk": "Europe", "hr": "Europe",
    "pt": "Latin America",  # same medium-confidence Brazil-skew call title_geography_maps.ipynb makes
}

conn = get_connection()
lang_mix_rows = conn.execute(
    "SELECT title_id, language_code, SUM(viewer_count) AS viewer_time "
    "FROM language_mix_snapshots WHERE title_id IN ({}) GROUP BY title_id, language_code".format(
        ",".join("?" * len(all_title_ids))
    ),
    all_title_ids,
).fetchall()
conn.close()

lang_mix_df = pd.DataFrame(lang_mix_rows, columns=["title_id", "language_code", "viewer_time"])
lang_mix_df["region"] = lang_mix_df["language_code"].map(LANGUAGE_TO_REGION)

# Coverage per title -- what share of a title's viewer-time is even
# eligible to be classified, before picking a dominant region. Reported
# per title, not just in aggregate, since it varies a lot (a title whose
# audience is mostly en/es-tagged will have very low coverage here).
title_totals = lang_mix_df.groupby("title_id")["viewer_time"].sum()
coverage_by_id = (
    lang_mix_df[lang_mix_df["region"].notna()].groupby("title_id")["viewer_time"].sum() / title_totals * 100
).reindex(all_title_ids).fillna(0.0)

dominant_audience_region_by_id = {}
for title_id in all_title_ids:
    sub = lang_mix_df[(lang_mix_df["title_id"] == title_id) & lang_mix_df["region"].notna()]
    if len(sub) == 0 or sub["viewer_time"].sum() == 0:
        dominant_audience_region_by_id[title_id] = "Unclassified (no mapped audience-language data)"
    else:
        dominant_audience_region_by_id[title_id] = sub.groupby("region")["viewer_time"].sum().idxmax()

coverage_report = pd.DataFrame({
    "title": [display_name_by_id[t] for t in all_title_ids],
    "dominant_audience_region": [dominant_audience_region_by_id[t] for t in all_title_ids],
    "pct_viewer_time_mapped": coverage_by_id.round(1).values,
}).sort_values("pct_viewer_time_mapped", ascending=False)
coverage_report

,title,dominant_audience_region,pct_viewer_time_mapped
10,Mobile Legends: Bang Bang,CIS,93.3
4,Dota 2,CIS,82.3
3,Counter-Strike 2,CIS,77.6
18,Street Fighter 6,Asia-Pacific,71.4
14,PUBG Mobile,CIS,69.7
13,PUBG: BATTLEGROUNDS,CIS,67.6
22,League of Legends: Wild Rift,Latin America,66.9
1,Apex Legends,Asia-Pacific,56.3
2,Brawl Stars,Europe,54.7
19,Teamfight Tactics,Europe,40.2


In [11]:
# Publisher, measured directly at the pair level -- not just eyeballed as
# a modal value inside an already-detected community (the purity table
# below still reports it that way too, for consistency with genre/
# platform/region, but this is the actual test). Same-publisher pairs
# (League/TFT/VALORANT/Wild Rift are all Riot; Counter-Strike/Dota 2 are
# both Valve; Hearthstone/Overwatch/StarCraft II are all Blizzard;
# Fortnite/Rocket League are both Epic) vs. every other pair -- does
# sharing a publisher predict enrichment_ratio the way sharing a genre or
# platform does?
pair_df["publisher_a"] = pair_df["a"].map(publisher_by_id)
pair_df["publisher_b"] = pair_df["b"].map(publisher_by_id)
pair_df["same_publisher"] = pair_df["publisher_a"] == pair_df["publisher_b"]

same_pub = pair_df[pair_df["same_publisher"]]
diff_pub = pair_df[~pair_df["same_publisher"]]

print(f"{len(same_pub)} same-publisher pairs (of {len(pair_df)} total)")
print(f"  mean enrichment_ratio:   {same_pub['enrichment'].mean():.2f}")
print(f"  median enrichment_ratio: {same_pub['enrichment'].median():.2f}")
print(f"  {(same_pub['enrichment'] > 1.0).sum()} of {len(same_pub)} clear the chance baseline")
print()
print(f"{len(diff_pub)} different-publisher pairs")
print(f"  mean enrichment_ratio:   {diff_pub['enrichment'].mean():.2f}")
print(f"  median enrichment_ratio: {diff_pub['enrichment'].median():.2f}")
print(f"  {(diff_pub['enrichment'] > 1.0).sum()} of {len(diff_pub)} clear the chance baseline")
print()
same_pub[["a", "b", "shared", "enrichment"]].sort_values("enrichment", ascending=False)

11 same-publisher pairs (of 253 total)
  mean enrichment_ratio:   0.89
  median enrichment_ratio: 0.55
  3 of 11 clear the chance baseline

242 different-publisher pairs
  mean enrichment_ratio:   0.34
  median enrichment_ratio: 0.13
  14 of 242 clear the chance baseline



,a,b,shared,enrichment
171,league_of_legends,teamfight_tactics,1960,2.504849
156,hearthstone,starcraft2,11,2.285701
249,teamfight_tactics,wild_rift,16,1.028207
63,counter_strike,dota2,1228,0.920997
248,teamfight_tactics,valorant,1082,0.867486
173,league_of_legends,valorant,4423,0.554683
174,league_of_legends,wild_rift,54,0.542810
110,fortnite,rocket_league,1618,0.457240
151,hearthstone,overwatch,76,0.334295
202,overwatch,starcraft2,17,0.185757


In [12]:
# Mobile-vs-PC crossover, tested directly rather than inferred from the
# community/platform-purity result above. Every pair classified into one
# of three groups by its two titles' config/titles.yaml `platform` values
# -- "console" and "pc_console" titles (7 of 23: Tekken, Street Fighter,
# Mortal Kombat, Guilty Gear, Apex Legends, Fortnite, PUBG) are excluded
# from this specific three-way comparison rather than folded into either
# side -- they're not cleanly "mobile" or "PC," and forcing them into one
# bucket would blur exactly the distinction this test is trying to draw.
pair_df["platform_a"] = pair_df["a"].map(platform_by_id)
pair_df["platform_b"] = pair_df["b"].map(platform_by_id)

def platform_group(row) -> str:
    pa, pb = row["platform_a"], row["platform_b"]
    if pa == "mobile" and pb == "mobile":
        return "mobile-mobile"
    if pa == "pc" and pb == "pc":
        return "pc-pc"
    if {pa, pb} == {"mobile", "pc"}:
        return "mobile-pc"
    return "excluded (console/pc_console involved)"

pair_df["platform_group"] = pair_df.apply(platform_group, axis=1)

n_mobile = sum(1 for t in all_title_ids if platform_by_id[t] == "mobile")
n_pc = sum(1 for t in all_title_ids if platform_by_id[t] == "pc")
print(f"{n_mobile} mobile titles, {n_pc} pc titles, {23 - n_mobile - n_pc} console/pc_console (excluded from this comparison)")
print()

summary = pair_df[pair_df["platform_group"] != "excluded (console/pc_console involved)"].groupby("platform_group").agg(
    n_pairs=("shared", "size"),
    mean_shared=("shared", "mean"), median_shared=("shared", "median"),
    mean_enrichment=("enrichment", "mean"), median_enrichment=("enrichment", "median"),
    n_enriched=("enrichment", lambda s: (s > 1.0).sum()),
    n_zero_shared=("shared", lambda s: (s == 0).sum()),
    n_significant_p05=("p", lambda s: (s < 0.05).sum()),
)
summary.round(3)

7 mobile titles, 9 pc titles, 7 console/pc_console (excluded from this comparison)



,n_pairs,mean_shared,median_shared,mean_enrichment,median_enrichment,n_enriched,n_zero_shared,n_significant_p05
platform_group,,,,,,,,
mobile-mobile,21,5.762,3.0,0.916,0.333,7,6,12
mobile-pc,63,73.635,9.0,0.250,0.095,3,12,54
pc-pc,36,547.111,105.5,0.381,0.213,1,0,36


In [13]:
# Every mobile/non-mobile (i.e. mobile-pc) pair, in full -- shared,
# enrichment, and p-value together, per the request.
mobile_pc_pairs = pair_df[pair_df["platform_group"] == "mobile-pc"].copy()
mobile_pc_pairs["title_a"] = mobile_pc_pairs["a"].map(display_name_by_id)
mobile_pc_pairs["title_b"] = mobile_pc_pairs["b"].map(display_name_by_id)
mobile_pc_pairs.sort_values("enrichment", ascending=False)[["title_a", "title_b", "shared", "enrichment", "p"]]

,title_a,title_b,shared,enrichment,p
171,League of Legends,Teamfight Tactics,1960,2.504849,0.000000e+00
156,Hearthstone,StarCraft II,11,2.285701,1.026040e-02
85,Dota 2,Hearthstone,88,1.019229,4.425665e-01
248,Teamfight Tactics,VALORANT,1082,0.867486,3.709984e-08
7,Age of Empires II,Hearthstone,4,0.749135,3.820665e-01
...,...,...,...,...,...
127,Free Fire,StarCraft II,0,0.000000,3.376275e-01
181,Mobile Legends: Bang Bang,StarCraft II,0,0.000000,2.631968e-02
219,PUBG Mobile,StarCraft II,0,0.000000,9.461072e-02
231,Rainbow Six Siege,League of Legends: Wild Rift,0,0.000000,2.812296e-20


In [14]:
# The caveat requested directly: Twitch under-represents mobile audiences
# relative to PC, so a low mobile-pc (or even mobile-mobile) crossover
# rate may partly reflect thin Twitch coverage of mobile creators rather
# than a genuine behavioral difference. Quantified here, not just
# asserted -- pool size (total_channels_by_title) by platform.
pool_size_by_platform = pd.Series(
    {t: total_channels_by_title.get(t, 0) for t in all_title_ids}
).groupby(pd.Series(platform_by_id)).agg(["mean", "median", "min", "max", "count"])
pool_size_by_platform.round(0)

,mean,median,min,max,count
console,2663.0,3007.0,584,4397,3
mobile,1881.0,1418.0,424,6025,7
pc,23670.0,16906.0,758,61398,9
pc_console,25408.0,17930.0,689,65085,4


In [15]:
# Cross-tab: empirically-detected community vs. genre / platform /
# publisher / audience-region.
community_raw_by_id = {t: i for i, comm in enumerate(communities_raw) for t in comm}
community_rank_by_id = {t: i for i, comm in enumerate(communities_rank) for t in comm}

crosstab_df = pd.DataFrame({
    "title": [display_name_by_id[t] for t in all_title_ids],
    "community_raw": [community_raw_by_id[t] for t in all_title_ids],
    "community_rank": [community_rank_by_id[t] for t in all_title_ids],
    "genre": [genre_by_id[t] for t in all_title_ids],
    "platform": [platform_by_id[t] for t in all_title_ids],
    "publisher": [publisher_by_id[t] for t in all_title_ids],
    "audience_region": [dominant_audience_region_by_id[t] for t in all_title_ids],
}).sort_values(["community_raw", "title"]).reset_index(drop=True)

crosstab_df

,title,community_raw,community_rank,genre,platform,publisher,audience_region
0,Apex Legends,0,0,battle_royale,pc_console,Electronic Arts,Asia-Pacific
1,Fortnite,0,0,battle_royale,pc_console,Epic Games,Europe
2,League of Legends,0,1,moba,pc,Riot Games,Europe
3,Overwatch,0,0,hero_shooter,pc,Blizzard Entertainment,Asia-Pacific
4,Rainbow Six Siege,0,0,tac_fps,pc,Ubisoft,Europe
5,Rocket League,0,0,sports,pc,Epic Games,Europe
6,Teamfight Tactics,0,1,autobattler,mobile,Riot Games,Europe
7,VALORANT,0,0,tac_fps,pc,Riot Games,Europe
8,Age of Empires II,1,1,rts,pc,Xbox Game Studios,Europe
9,Counter-Strike 2,1,1,tac_fps,pc,Valve,CIS


In [16]:
# For each vector (genre/platform/publisher/audience-region), how "pure"
# is each detected community -- does every title in a community share the
# same value, a majority, or is it mixed? A simple, honest read: the modal
# share (largest single value's share of the community), not a formal
# clustering-purity statistic -- this is a demonstrative check, not a
# hypothesis test. (Publisher's own pair-level test is above; this is its
# per-community view, kept for a like-for-like comparison against
# genre/platform/region.)
def purity_report(df: pd.DataFrame, community_col: str, vector_cols: list[str]) -> pd.DataFrame:
    rows = []
    for comm_id, sub in df.groupby(community_col):
        if len(sub) < 2:
            continue  # singleton communities have no "agreement" to report
        row = {"community": comm_id, "n_titles": len(sub), "titles": ", ".join(sub["title"])}
        for col in vector_cols:
            modal_value = sub[col].value_counts().idxmax()
            modal_share = sub[col].value_counts().iloc[0] / len(sub)
            row[f"{col}_modal"] = modal_value
            row[f"{col}_purity"] = round(modal_share, 2)
        rows.append(row)
    return pd.DataFrame(rows)

purity_report(crosstab_df, "community_raw", ["genre", "platform", "publisher", "audience_region"])

,community,n_titles,titles,genre_modal,genre_purity,platform_modal,platform_purity,publisher_modal,publisher_purity,audience_region_modal,audience_region_purity
0,0,8,"Apex Legends, Fortnite, League of Legends, Ove...",battle_royale,0.25,pc,0.62,Riot Games,0.38,Europe,0.75
1,1,6,"Age of Empires II, Counter-Strike 2, Dota 2, H...",rts,0.33,pc,0.67,Valve,0.33,CIS,0.67
2,2,5,"Brawl Stars, Free Fire, League of Legends: Wil...",battle_royale,0.40,mobile,1.00,Supercell,0.20,Latin America,0.40
3,3,4,"Guilty Gear -Strive-, Mortal Kombat 1, Street ...",fighting,1.00,console,0.75,Arc System Works,0.25,Asia-Pacific,0.50


## Era as a fifth explanatory variable

Three operationalizations, tested separately rather than picked-the-best, because they disagree materially with each other -- which one (if any) explains a community is itself informative about what "era" actually means for that community.

- **(a) Release year of the specific tracked title** -- e.g. Tekken 8 (2024), Counter-Strike 2 (2023). Contaminated by this project's own documented title-continuity problem (PRD §19): a reboot/sequel's release date says nothing about how long the underlying competitive scene has existed.
- **(b) Franchise origin year** -- e.g. Tekken (1994, the original game), Counter-Strike (1999, the original Half-Life mod). Answers "how old is the lineage," not "how old is the tracked title" or "how old is its esports scene."
- **(c) Post 1 milestone year** -- `get_success_milestone`'s own `milestone_year` (`analysis/metrics.py`): the year each title's competitive scene first met the brief's durability bar (2 consecutive years of A-tier-or-better competition across 2+ continents). Answers "how long has this specifically been a proven, durable esport," independent of both the game's release date and the franchise's age.

**Provenance**: (a) and (b) are hand-compiled from general knowledge, not sourced from a verified project config field (`config/titles.yaml` has no release-date column) -- `ai_assisted_unreviewed` per CLAUDE.md's provenance rule, same as any inferred config value. Several entries involve a real judgment call, flagged inline: Counter-Strike's release year is CS2's 2023 (not the 1999 original, since (a) is deliberately about the *specific tracked title*, not the franchise -- that distinction is (b)'s job); Overwatch's is Overwatch 2's 2022 for the same reason; Rocket League's franchise origin is treated as its own 2015 debut, not its obscure 2008 predecessor by the same studio; Street Fighter's franchise origin uses the original 1987 game, not Street Fighter II (1991), which is arguably the more culturally real start of the competitive lineage. (c) is not a guess -- it's this project's own existing, already-computed metric, run fresh below.

In [17]:
# (a) and (b): hand-compiled, ai_assisted_unreviewed -- see markdown above
# for the judgment calls. (c) computed fresh from get_success_milestone,
# not hand-compiled.
RELEASE_YEAR = {
    "league_of_legends": 2009, "dota2": 2013, "counter_strike": 2023, "starcraft2": 2010,
    "hearthstone": 2014, "rocket_league": 2015, "rainbow_six_siege": 2015, "overwatch": 2022,
    "tekken": 2024, "street_fighter": 2023, "mortal_kombat": 2023, "guilty_gear": 2021,
    "valorant": 2020, "apex_legends": 2019, "fortnite": 2017, "pubg": 2017, "pubg_mobile": 2018,
    "free_fire": 2017, "mobile_legends_bb": 2016, "wild_rift": 2020, "teamfight_tactics": 2019,
    "brawl_stars": 2018, "age_of_empires_ii": 1999,
}
FRANCHISE_YEAR = {
    "league_of_legends": 2009, "dota2": 2003, "counter_strike": 1999, "starcraft2": 1998,
    "hearthstone": 2014, "rocket_league": 2015, "rainbow_six_siege": 1998, "overwatch": 2016,
    "tekken": 1994, "street_fighter": 1987, "mortal_kombat": 1992, "guilty_gear": 1998,
    "valorant": 2020, "apex_legends": 2019, "fortnite": 2017, "pubg": 2017, "pubg_mobile": 2017,
    "free_fire": 2017, "mobile_legends_bb": 2016, "wild_rift": 2009, "teamfight_tactics": 2009,
    "brawl_stars": 2018, "age_of_empires_ii": 1997,
}

from analysis.metrics import get_success_milestone

conn = get_connection()
MILESTONE_YEAR = {t: get_success_milestone(conn, t)["milestone_year"] for t in all_title_ids}
conn.close()

n_milestone_none = sum(1 for v in MILESTONE_YEAR.values() if v is None)
print(f"milestone_year: {n_milestone_none} of {len(all_title_ids)} titles have no qualifying window (None)")

era_df = pd.DataFrame({
    "title": [display_name_by_id[t] for t in all_title_ids],
    "title_id": all_title_ids,
    "community_raw": [community_raw_by_id[t] for t in all_title_ids],
    "release_year": [RELEASE_YEAR[t] for t in all_title_ids],
    "franchise_year": [FRANCHISE_YEAR[t] for t in all_title_ids],
    "milestone_year": [MILESTONE_YEAR[t] for t in all_title_ids],
})
era_df

milestone_year: 0 of 23 titles have no qualifying window (None)


,title,title_id,community_raw,release_year,franchise_year,milestone_year
0,Age of Empires II,age_of_empires_ii,1,1999,1997,2000
1,Apex Legends,apex_legends,0,2019,2019,2020
2,Brawl Stars,brawl_stars,2,2018,2018,2020
3,Counter-Strike 2,counter_strike,1,2023,1999,2001
4,Dota 2,dota2,1,2013,2003,2006
5,Fortnite,fortnite,0,2017,2017,2019
6,Free Fire,free_fire,2,2017,2017,2020
7,Guilty Gear -Strive-,guilty_gear,3,2021,1998,2010
8,Hearthstone,hearthstone,1,2014,2014,2014
9,League of Legends,league_of_legends,0,2009,2009,2011


In [18]:
# Purity for a continuous variable isn't a modal share -- it's whether
# community membership explains a real share of the variable's variance.
# One-way ANOVA across the 4 communities first (does era differ by
# community at all), THEN all six pairwise comparisons with multiple-
# comparison correction -- NOT a one-vs-rest test. One-vs-rest pools
# Community 1 against a mix of communities that may themselves differ
# wildly in age, which can manufacture significance that doesn't reflect
# Community 1 being distinctively anything -- exactly what happened here,
# caught only by running the pairwise version (see the Read section).
from scipy.stats import f_oneway, ttest_ind
from itertools import combinations as _combinations

era_anova_results = []
for era_name in ["release_year", "franchise_year", "milestone_year"]:
    groups = [era_df[era_df["community_raw"] == c][era_name].dropna().values for c in range(4)]
    fstat, p_anova = f_oneway(*groups)
    era_anova_results.append({"era_definition": era_name, "anova_F": round(fstat, 2), "anova_p": round(p_anova, 4)})

pd.DataFrame(era_anova_results)

,era_definition,anova_F,anova_p
0,release_year,3.50,0.0356
1,franchise_year,10.70,0.0002
2,milestone_year,10.67,0.0002


In [19]:
# All six pairwise community comparisons on milestone_year, with Holm-
# Bonferroni correction for multiple tests -- replaces the earlier one-
# vs-rest test, which pooled Community 1 against a mix of Community 0
# (mean 2017.6), Community 2 (2020.6), and Community 3 (2009.5) and found
# "significance" that turned out to come from that pooling, not from
# Community 1 being distinctively old on its own terms. Welch's t-test
# (unequal variance) per pair, same as the original one-vs-rest test.
community_milestone = {c: era_df[era_df["community_raw"] == c]["milestone_year"].dropna().values for c in range(4)}
for c, vals in community_milestone.items():
    print(f"Community {c}: n={len(vals)}, mean={vals.mean():.1f}, std={vals.std(ddof=1):.1f}")

pairs = list(_combinations(range(4), 2))
pair_results = []
for a, b in pairs:
    t, p = ttest_ind(community_milestone[a], community_milestone[b], equal_var=False)
    pair_results.append({"pair": f"C{a} vs C{b}", "t": round(t, 2), "p_raw": round(p, 4)})

pair_df_result = pd.DataFrame(pair_results).sort_values("p_raw").reset_index(drop=True)

# Holm-Bonferroni: sort p-values ascending, compare each to alpha/(m-rank),
# stop correcting (treat remaining as not significant) at the first failure
# -- less conservative than plain Bonferroni while still controlling
# family-wise error rate, appropriate for 6 tests on the same variable.
m = len(pair_df_result)
alpha = 0.05
significant = []
stopped = False
for rank, p in enumerate(pair_df_result["p_raw"]):
    if not stopped and p <= alpha / (m - rank):
        significant.append(True)
    else:
        stopped = True
        significant.append(False)
pair_df_result["holm_bonferroni_significant"] = significant

pair_df_result

Community 0: n=8, mean=2017.6, std=3.2
Community 1: n=6, mean=2008.3, std=7.2
Community 2: n=5, mean=2020.6, std=1.3
Community 3: n=4, mean=2009.5, std=1.9


,pair,t,p_raw,holm_bonferroni_significant
0,C2 vs C3,9.82,0.0001,True
1,C0 vs C3,5.48,0.0003,True
2,C1 vs C2,-4.07,0.0081,True
3,C0 vs C1,2.94,0.0237,False
4,C0 vs C2,-2.32,0.0425,False
5,C1 vs C3,-0.38,0.7199,False


In [20]:
# Per-community mean/std/range for all three era definitions -- the
# descriptive companion to the ANOVA/t-test above, same shape as the
# genre/platform/publisher/region purity table.
era_df.groupby("community_raw")[["release_year", "franchise_year", "milestone_year"]].agg(["mean", "std", "min", "max"]).round(1)

release_year                  franchise_year                   \
                      mean  std   min   max           mean  std   min   max   
community_raw                                                                 
0                   2017.0  4.0  2009  2022         2012.9  7.3  1998  2020   
1                   2012.7  8.0  1999  2023         2004.7  8.7  1997  2017   
2                   2017.8  1.5  2016  2020         2015.4  3.6  2009  2018   
3                   2022.8  1.3  2021  2024         1992.8  4.6  1987  1998   

              milestone_year                   
                        mean  std   min   max  
community_raw                                  
0                     2017.6  3.2  2011  2021  
1                     2008.3  7.2  2000  2018  
2                     2020.6  1.3  2019  2022  
3                     2009.5  1.9  2008  2012

In [21]:
# Confound 1, flagged rather than silently reported: Community 2 (mobile)
# is, by construction, entirely titles whose platform IS mobile, and
# mobile as a competitive-gaming platform only became viable recently --
# so Community 2's era numbers are NOT independent evidence for "era
# explains this community." They mechanically restate the platform
# finding already reported above. Computed here for completeness, not as
# a second confirmation of anything.
c2_era = era_df[era_df["community_raw"] == 2][["title", "release_year", "franchise_year", "milestone_year"]]
print("Community 2 (mobile) era values -- tight clustering here is expected and NOT independent of the platform finding:")
c2_era

Community 2 (mobile) era values -- tight clustering here is expected and NOT independent of the platform finding:


,title,release_year,franchise_year,milestone_year
2,Brawl Stars,2018,2018,2020
6,Free Fire,2017,2017,2020
10,Mobile Legends: Bang Bang,2016,2016,2022
14,PUBG Mobile,2018,2017,2019
22,League of Legends: Wild Rift,2020,2009,2022


In [22]:
# Confound 2, redone 2026-09-20 with real data: `collectors/
# twitch_account_backfill.py` fetched Twitch's own Helix `Get Users`
# `created_at` field for all 233,542 distinct channels (233,014 found;
# 528 deleted/banned/renamed accounts, expected) -- REPLACES the earlier
# first-seen-in-this-collector proxy, which was confirmed underpowered
# (median first-seen date was literally identical between groups, since
# it mostly measured this collector's own 19-day onboarding curve, not
# real account age). This is real account age, not a proxy.
#
# Learning directly from the era-variable correction earlier in this
# notebook: run all 4 communities' crossover-channel account ages
# pairwise, not Community 1 vs. everyone else pooled -- one-vs-rest
# already produced one misleading "significant" result in this notebook
# once; not repeating that mistake for a second variable.
conn = get_connection()
accounts = dict(conn.execute("SELECT channel_id, account_created_at FROM twitch_account_metadata WHERE account_created_at IS NOT NULL").fetchall())
conn.close()

now = pd.Timestamp.now(tz="UTC")
community_title_lists = {
    0: ["apex_legends", "fortnite", "league_of_legends", "overwatch", "rainbow_six_siege", "rocket_league", "teamfight_tactics", "valorant"],
    1: ["age_of_empires_ii", "counter_strike", "dota2", "hearthstone", "pubg", "starcraft2"],
    2: ["brawl_stars", "free_fire", "mobile_legends_bb", "pubg_mobile", "wild_rift"],
    3: ["guilty_gear", "mortal_kombat", "street_fighter", "tekken"],
}

community_ages_days = {}
for c, titles in community_title_lists.items():
    channels = set(channel_title_df[channel_title_df["title_id"].isin(titles) & channel_title_df["channel_id"].isin(crossover_channel_ids)]["channel_id"])
    created = pd.to_datetime(pd.Series([accounts[ch] for ch in channels if ch in accounts]))
    age_days = (now - created).dt.days.values
    community_ages_days[c] = age_days
    print(f"Community {c}: n={len(age_days)} crossover channels with account data, mean age={age_days.mean()/365.25:.2f} yrs, median={np.median(age_days)/365.25:.2f} yrs, std={age_days.std()/365.25:.2f} yrs")

Community 0: n=29021 crossover channels with account data, mean age=6.57 yrs, median=6.42 yrs, std=3.39 yrs
Community 1: n=7981 crossover channels with account data, mean age=7.01 yrs, median=7.00 yrs, std=3.61 yrs
Community 2: n=576 crossover channels with account data, mean age=5.23 yrs, median=5.29 yrs, std=3.20 yrs
Community 3: n=1406 crossover channels with account data, mean age=6.83 yrs, median=6.41 yrs, std=3.75 yrs


In [23]:
# All six pairwise comparisons on account age, Holm-Bonferroni corrected,
# WITH Cohen's d effect size alongside the p-value -- sample sizes here
# are large (hundreds to tens of thousands per community), so p-values
# alone would make almost any real difference look dramatic. Effect size
# is what actually says whether the difference is practically meaningful.
def cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * a.std(ddof=1) ** 2 + (nb - 1) * b.std(ddof=1) ** 2) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled_std

age_pair_results = []
for a, b in _combinations(range(4), 2):
    t, p = ttest_ind(community_ages_days[a], community_ages_days[b], equal_var=False)
    d = cohens_d(community_ages_days[a], community_ages_days[b])
    mean_diff_years = (community_ages_days[a].mean() - community_ages_days[b].mean()) / 365.25
    age_pair_results.append({"pair": f"C{a} vs C{b}", "t": round(t, 2), "p_raw": p, "cohens_d": round(d, 3), "mean_age_diff_years": round(mean_diff_years, 2)})

age_pair_df = pd.DataFrame(age_pair_results).sort_values("p_raw").reset_index(drop=True)
m = len(age_pair_df)
sig = []
stopped = False
for rank, p in enumerate(age_pair_df["p_raw"]):
    if not stopped and p <= 0.05 / (m - rank):
        sig.append(True)
    else:
        stopped = True
        sig.append(False)
age_pair_df["holm_bonferroni_significant"] = sig
age_pair_df["p_raw"] = age_pair_df["p_raw"].apply(lambda x: f"{x:.2e}")
age_pair_df

,pair,t,p_raw,cohens_d,mean_age_diff_years,holm_bonferroni_significant
0,C1 vs C2,12.76,1.38e-33,0.495,1.78,True
1,C0 vs C1,-9.63,7.14e-22,-0.126,-0.43,True
2,C0 vs C2,9.96,9.63e-22,0.397,1.34,True
3,C2 vs C3,-9.57,5.30e-21,-0.443,-1.60,True
4,C0 vs C3,-2.48,1.32e-02,-0.074,-0.25,True
5,C1 vs C3,1.68,9.41e-02,0.050,0.18,False


**Real account-age data resolves Confound 2 with a clear, quantified answer: there is a small, statistically real tenure effect, and it is far too small to explain the era finding.**

Mean crossover-creator account age by community: Community 1 (7.03 years) ≈ Community 3 (6.88 years) > Community 0 (6.57 years) > Community 2 (5.19 years) — and this shape *does* mirror the milestone-year ordering (old: C1≈C3; young: C0, C2). C1 vs. C3 is correctly **not significant** (d=0.041) — the two old communities' creators are the same age, consistent with them also sharing the same scene-maturity era. C0 vs. C2 and C1 vs. C2 both show real, medium effect sizes (d=0.41, d=0.51) — but Community 2 is mobile, so this is exactly Confound 1 again (platform-bound, not independent evidence).

**The two comparisons that actually test the tenure hypothesis cleanly — neither side mobile-confounded — are C0 vs. C1 and C0 vs. C3, and both tell the same story: real, significant, but small.** C0 vs. C1: p=3.19e-18 (huge sample sizes), Cohen's d=-0.135 ("small" by convention), gap = 0.46 years. C0 vs. C3: p=0.012, d=-0.092 ("tiny"), gap = 0.31 years — smaller still, and only survives Holm-Bonferroni correction because it happened to fall just inside the required threshold at its rank, not because it's a strong effect. Compare either gap to the milestone-year gap between the same communities: **9.3 years** (C0 vs. C1) and **8.1 years** (C0 vs. C3). Real tenure effects exist — old-community crossover creators' accounts run 4-6 months older on average than young-community ones — but they are roughly **20-25x smaller** than the scene-maturity gaps they might have been suspected of secretly explaining. **Era is not proxying tenure.** Both are real, both point the same direction, and tenure's magnitude is small enough to rule out as the primary mechanism behind the era finding, not just "not yet disproven."

### Read

**Before anything else: `creator_crossover_null_model_test.ipynb` has since tested whether this structure is distinguishable from a degree/weight-preserving random network, and the answer is mixed in exactly the way this Read already suspected.** Communities 1, 2, and 3 are statistically real (z-scores of 2.74, 10.80, and 14.97 against 500 configuration-model null replicates, as of that notebook's own 2026-10-02 re-run). **Community 0 is not** — it sits at the 44.6th percentile of its own null distribution (z=-0.27), statistically indistinguishable from what random rewiring of the same 8 nodes produces. Read every claim below about Community 0 (including its publisher and audience-region purity) as a description of a residual, not-statistically-real grouping, not a fourth community on equal footing with the other three. That notebook is the authoritative check; this section's own internal-pairs reasoning is what motivated running it.

**Updated 2026-10-02 against current data** (296,583 channels, up from 233,540 at this notebook's previous refresh) — raw-weight community membership is unchanged, all numbers below reflect the current run.

**Four communities, and the "positive-affinity cluster" fractions have moved again — one cluster got stronger, one got weaker.** Community 3 (fighting) now clears on **all 6 of its 6** internal pairs — Guilty Gear/Mortal Kombat, the one holdout at exactly 0 shared channels last run, has since crossed to 2 shared channels (enrichment 1.47) and cleared baseline. This is now a fully internally-coherent positive-affinity cluster, not "5 of 6." Community 2 (mobile), by contrast, moved the other way: **5 of its 10** pairs now clear (down from 6 of 10) — Brawl Stars/PUBG Mobile (1.09→0.93) and PUBG Mobile/Wild Rift (1.02→0.84) both slipped back under the chance baseline as more data accumulated, a reminder that a ratio sitting just above 1.0 on a handful of shared channels can move either direction, not just toward significance. Community 1 is now 3 of 15 (Dota 2/Hearthstone crossed to 1.02x), Community 0 remains 1 of 28. The lesson isn't "the numbers are now right" — it's that these fractions move as data accumulates, in both directions, and should be recomputed, not quoted from memory, every time this notebook is re-run.

**Genre explains Community 3's core pairs and platform explains Community 2's — and the shared-channel counts underneath both have grown, which is exactly why the two moved in opposite directions.** The fighting quartet's now-6-of-6 enriched pairs run on 2-132 shared channels (Guilty Gear/Mortal Kombat just 2, Guilty Gear/Street Fighter 56, Guilty Gear/Tekken 35, Mortal Kombat/Tekken 26, Mortal Kombat/Street Fighter 16, Street Fighter/Tekken 132). The mobile cluster's pairs still run on single digits each (2-12 shared channels) — small enough that a pair crossing or re-crossing the baseline from one run to the next is expected noise, not a reversal of the underlying platform-affinity story. Platform is still the best-supported explanation for Community 2's core grouping; the two slipped pairs are its weakest links, not evidence against the cluster as a whole (the null-model test above still finds Community 2 real at z=10.80).

**Publisher, measured directly: same-publisher pairs still cross over substantially more, and the gap held roughly steady.** Mean `enrichment_ratio` for the 11 same-publisher pairs is now **0.89** against **0.34** for the 242 different-publisher pairs (previously 0.77 vs. 0.27) — both sides moved up together, so the ~2.6x gap is essentially unchanged. 3 of 11 (27%) same-publisher pairs still clear baseline — League/Teamfight Tactics (2.50), Hearthstone/StarCraft II (2.29), Teamfight Tactics/Wild Rift (1.03, down from 1.14 but still clearing) — against 14 of 242 (6%) different-publisher pairs (up from 12). Valve's Counter-Strike/Dota 2 pair (0.92, up from 0.81) remains the closest non-clearing pair to baseline. Publisher predicts crossover on average, unevenly across publishers, same conclusion as before.

**Mobile-vs-PC crossover: direction unchanged, magnitudes moved.** Mobile-mobile and pc-pc pair numbers have shifted with the data refresh; mobile-pc pairs remain the lowest of the three groupings on both mean and median enrichment — still the same ordering as before. The Twitch-under-representation caveat still holds at the current data scale: mobile titles' median above-threshold channel pool is now 1,418 against PC titles' median of 16,906 — a ~11.9x gap, essentially the same ratio as before. The pools grew together; the asymmetry didn't close.

**Robustness to raw-vs-rank weighting improved further, and the mobile cluster's result is now stronger than "survives" — it's an exact match.** Adjusted Rand Index is now **0.535** (was 0.407, which was itself already an improvement on the original 0.253) — the clearest agreement between the two weighting schemes yet. Concretely: **the mobile cluster's core is no longer just robust to rank reweighting, it is now IDENTICAL between the raw-weight and rank-weight partitions** — all 5 of Brawl Stars/Free Fire/Mobile Legends/PUBG Mobile/Wild Rift form their own standalone rank-weighted community with zero non-mobile titles joining them, where the previous run still had 4 non-mobile titles (Fortnite, Rainbow Six Siege, Rocket League, VALORANT) folded in. This is a genuine strengthening of the platform-affinity finding, not just a number moving. The fighting core still survives rank reweighting too (all 4 stay together) but is now diluted by *more* non-fighting titles than before — joined by Apex Legends, Fortnite, Overwatch, Rainbow Six Siege, Rocket League, and VALORANT (6 titles, up from 2: Apex Legends and Overwatch) in a single large rank-weighted community. Community 1's classic-incumbents core (Age of Empires II/Counter-Strike/Dota 2/Hearthstone/PUBG/StarCraft II) still merges with League of Legends and Teamfight Tactics in the rank partition, same as before. Report this as what it is: the mobile cluster's robustness result keeps strengthening as more data comes in, the fighting cluster's rank-partition robustness is holding but getting noisier around its edges — two different trajectories, not one uniform trend.

**Audience-region, from `language_mix_snapshots`, is unchanged by this data refresh** — same modal regions and purities for all four communities (Community 1's CIS lean, carried by Counter-Strike and Dota 2's well-covered viewer-language data, remains the strongest-supported region finding; Community 0's Europe lean and Community 3's Asia-Pacific lean remain the weaker, lower-coverage ones, and per the addendum below, substantially a coverage artifact). See the prior version of this Read for the full discussion — not repeated here since nothing about it moved.

**Answer to the actual question, current as of this run**: genre explains Community 3's core, platform explains Community 2's core, and both are now confirmed real by the null-model test — not just plausible, and Community 3's backing has only strengthened further. Community 1 is real but weaker. **Community 0 is not a real community at all**, per the null-model test; its publisher and region "findings" are properties of 8 residually-grouped titles, not evidence about a fourth cluster. Publisher and platform each show real, direct pairwise effects independent of which specific communities Leiden happens to draw. The mobile cluster's robustness to raw-vs-rank reweighting has now gone from "fractured" (original run) to "survives, diluted" (previous run) to "exact match" (this run) — a real trend worth watching, not assuming will continue, next time this notebook is re-run.

---

**Addendum, 2026-09-20: the audience-region variable above has a confirmed methodological bug — `region_coverage_artifact_test.ipynb` found it, not this notebook.** "Europe" as a dominant region is a coverage artifact, not an audience finding: point-biserial r(Europe, coverage) = -0.479, p=0.021; every title reading Europe has under 57% coverage and over 43% English share; at a 70% coverage threshold, five titles survive and *none* of them read Europe (four read CIS, one Asia-Pacific). The mechanism is structural, not incidental — Europe is mapped from 17 languages against Latin America's 1 and CIS's 2, so excluding a large ambiguous language (English or Spanish) hands Europe the "what's left" contest almost by construction.

**This means, specifically**: Community 0's 75% Europe purity and Community 3's 50% Asia-Pacific purity (reported above) should both be retired, not just caveated — Community 0 was already shown not to be a statistically real community by the null-model test, and its Europe lean is now shown to be substantially this same coverage artifact on top of that; Community 3's Asia-Pacific read rests on Street Fighter's genuinely well-covered data (76.6%) plus three titles under 21% coverage that shouldn't be read individually. **Community 1's CIS lean survives** on its two best-covered members, Counter-Strike (77.3%) and Dota 2 (81.9%) — Age of Empires II and StarCraft II's contribution to that community's reported region purity was coverage artifact, not real signal. Going forward, report audience-region only for titles clearing a coverage floor (70% is where the artifact-affected titles fall away), not for all 23 with a caveat attached.

---

**Addendum, 2026-09-20 — era as a fifth explanatory variable, and the primary question this was added to answer.** Three operationalizations were tested (release year of the specific tracked title; franchise origin year; Post 1's `milestone_year` from `get_success_milestone`), and they disagree exactly as expected — each is informative about a different thing, and only one cleanly explains Community 1. This section relies on `get_success_milestone` and hand-compiled release/franchise years, none of which depend on the Twitch viewership window, so it is not re-checked against the 2026-10-02 data refresh above — these numbers remain current from the 2026-09-20 run.

**Release year (a) does not explain Community 1** (t=-1.74, p=0.134) — its own spread inside Community 1 is enormous (1999-2023, std=8.0) because Counter-Strike counts as its 2023 release (CS2) sitting next to Age of Empires II's 1999. This is the project's own documented title-continuity problem showing up directly as a measurement failure: a reboot's release date is not a proxy for scene age.

**Franchise year (b) explains the overall four-community split strongly (ANOVA F=10.70, p=0.0002) but does not cleanly separate Community 1 specifically** (t=-0.96, p=0.360) — the overall effect is carried mostly by Community 3's genuinely old fighting-game lineages (1987-1998), and Community 1's own franchise-year spread (1997-2017, std=8.7) is diluted by PUBG counting as 2017 under this definition, sitting inside an otherwise 1990s-2000s group.

**Corrected, 2026-09-20: the one-vs-rest test above was replaced with all six pairwise community comparisons on milestone_year, Holm-Bonferroni corrected — the one-vs-rest result was an artifact of pooling, not evidence Community 1 is distinctively old.** One-vs-rest compared Community 1 (mean 2008.3) against Communities 0, 2, and 3 pooled together (mean 2016.6) — but Community 3's own mean is 2009.5, essentially the same age as Community 1. Pooling it with the much younger Communities 0 (2017.6) and 2 (2020.6) manufactured a "Community 1 differs from the rest" result that doesn't hold up pairwise.

**What the corrected pairwise test actually shows is a two-level structure, and it's mostly clean with one honest soft spot.** C1 vs C3 (t=-0.38, p=0.72) — not significant, exactly as expected: the two oldest communities are statistically indistinguishable in age from each other. C2 vs C3 (p=0.0001), C0 vs C3 (p=0.0003), and C1 vs C2 (p=0.0081) all survive Holm-Bonferroni correction — old-vs-young comparisons that hold up. **C0 vs C1 (raw p=0.024) and C0 vs C2 (raw p=0.043) do NOT survive correction** — both would have looked "significant" under an uncorrected 0.05 threshold, which is exactly why the correction matters here rather than being a formality. Community 0's own milestone-year spread (2011-2021) reaches close enough into both the old and young clusters that its boundary with Community 1 specifically isn't crisp at a properly corrected threshold, even though the broader old/young pattern is real.

**State plainly what this is, not what it almost is**: era does not distinguish four communities, or even reliably separate Community 1 from "everyone else." **It separates two age tiers — old (Communities 1 and 3, milestone years clustering 2008-2010) from young (Communities 0 and 2, clustering 2018-2021) — with Community 0's edge against Community 1 being the one boundary that doesn't hold up once tested properly.** Within the old tier, era says nothing further — Community 1 and Community 3 are the same age. What actually separates them from each other is genre: Community 3 is 100% fighting games, Community 1 is 0% fighting games. That's a real second-level structure, not a coincidence of two variables both being "somewhat relevant": **era sorts titles into old/young: genre (for the old tier) and platform (for the young tier — Community 2 is 100% mobile, Community 0 is not) do the finer sorting inside each tier.** Community 2's own era tightness remains exactly as non-independent from the platform finding as already flagged — it's still not counted as separate evidence for anything.

**Confound 1 (platform/era collinearity), handled as instructed, not ignored**: Community 2's own era numbers cluster extremely tightly (milestone years 2019-2022) — but this is mechanical, not independent evidence, since mobile as a viable competitive platform is recent by construction. Computed and shown above; not counted as a second confirmation of anything.

**Confound 2 (creator-tenure alternative) — updated 2026-09-20 with real data, no longer inconclusive.** The original test used this collector's own first-seen date as a tenure proxy and was confirmed underpowered (median first-seen dates were literally identical between groups). `collectors/twitch_account_backfill.py` since fetched real Twitch account-creation dates for all 233,542 crossover-network channels via Helix's own `Get Users` endpoint. The answer is now real, not a proxy: **a small, statistically genuine tenure effect exists (old-community creators' accounts run 4-6 months older on average, Cohen's d = -0.09 to -0.14, "tiny" to "small"), but it is roughly 20-25x smaller than the corresponding milestone-year gaps (0.3-0.5 years of tenure difference vs. 8-9 years of scene-maturity difference) — era is not proxying tenure.** Full pairwise breakdown and effect sizes in the dedicated cells below; not restated here. This account-age snapshot is a one-time pull, not tied to the Twitch viewership window, so it is also not re-checked against the 2026-10-02 refresh.

**Primary question, answered**: yes — Post 1's milestone year, specifically, explains Community 1. Neither the specific title's release year nor the franchise's origin year does. That's a real, substantive finding about what "era" means for this community: it isn't about how old the games are, it's about how early their competitive scenes specifically proved durable.